# Homework 5 —  Starter Templet
Run locally in Jupyter with a GLFW-capable desktop environment. Complete TODOs 1–6 in the code cell.


In [ ]:
import sys
import os
import ctypes
import numpy as np
from PIL import Image
import glfw
from OpenGL.GL import *

# ==============================================================================
# 1. GLSL Shader Source Code
# ==============================================================================

VERTEX_SHADER_SRC = """#version 330 core
layout(location = 0) in vec3 aPos;  // Vertex 3D position
layout(location = 1) in vec2 aUv;   // 2D UV texture coordinate

uniform mat4 uModelView;
uniform mat4 uPersp;

out vec3 vPoint; // Interpolated camera/eye-space position
out vec2 vUv;    // Interpolated UV coordinates passed to fragment shader

void main() {
    // Transform vertex position into camera space
    vPoint = (uModelView * vec4(aPos, 1.0)).xyz;

    // Project camera-space coordinates into 4D clip space
    gl_Position = uPersp * vec4(vPoint, 1.0);

    // Pass UV coordinates directly down to rasterizer
    vUv = aUv;
}
"""

FRAGMENT_SHADER_SRC = """#version 330 core
in vec3 vPoint;
in vec2 vUv;

out vec4 FragColor;

uniform sampler2D uTexture; // 2D texture sampler linked to texture unit 0

// Multiple static lights passed in camera/eye space
uniform int uNumLights;
uniform vec3 uLights[8];

// Illumination coefficient weights
uniform float uAmb = 0.20;  // Ambient reflection weight
uniform float uDif = 0.70;  // Diffuse reflection weight
uniform float uSpc = 0.50;  // Specular reflection weight

void main() {
    // 1. Analytical face normal using screen-space partial derivatives (faceted appearance)
    vec3 dx = dFdx(vPoint);
    vec3 dy = dFdy(vPoint);
    vec3 N = normalize(cross(dx, dy));

    // Eye vector: camera sits at origin (0, 0, 0) in camera space
    vec3 E = normalize(-vPoint);

    // 2. Accumulate lighting contributions across all static lights
    float totalDiffuse = 0.0;
    float totalSpecular = 0.0;

    for (int i = 0; i < uNumLights; i++) {
        // Light vector pointing from surface to light source
        vec3 L = normalize(uLights[i] - vPoint);

        // Lambertian diffuse component
        float diff = max(0.0, dot(N, L));
        totalDiffuse += diff;

        // Phong specular highlight
        vec3 R = reflect(-L, N);
        float specAngle = max(0.0, dot(R, E));
        if (diff > 0.0) totalSpecular += pow(specAngle, 32.0);
    }
    // 3. multiplies lighting scalar across the RGB vector of the texture
    vec3 texColor = texture(uTexture, vUv).rgb;
    // 4. Combine ambient base with accumulated diffuse and specular terms
    float baseLight = min(1.0, uAmb + uDif * totalDiffuse);
    vec3 finalColor = baseLight * texColor + vec3(uSpc * totalSpecular);
    FragColor = vec4(clamp(finalColor, 0.0, 1.0), 1.0);

}
"""

# ==============================================================================
# 2. Geometry Builder with Automated UV Mapping
# ==============================================================================

def build_textured_letter_b():
    """
    Builds the 3D extruded letter B, mapping UV coordinates to [0, 1]^2
    using the bounding box of the front face.
    """
    # 2D front face vertices (symmetric spine at x = -80)
    front_2d = np.array([
        [  0.0,    0.0],  # 0: J (central hub)
        [ 50.0, -150.0],  # 1: E
        [ 80.0, -130.0],  # 2: F
        [ 80.0,  -50.0],  # 3: G
        [ 50.0,    0.0],  # 4: I
        [ 80.0,   50.0],  # 5: H
        [ 80.0,  130.0],  # 6: D
        [ 50.0,  150.0],  # 7: C
        [-50.0,  150.0],  # 8: A
        [-50.0, -150.0]   # 9: B
    ], dtype=np.float32)

    # 1. Generate UVs mapped to [0, 1] based on bounds before scaling
    min_xy = front_2d.min(axis=0)
    max_xy = front_2d.max(axis=0)
    # TODO 1 (UV mapping): normalize each XY coordinate to the [0,1] range.
    # Use min_xy and max_xy. Replace the placeholder below.
    uvs_front = np.full((len(front_2d), 2), 0.5, dtype=np.float32)
    # Both front and back faces share identical UV projections
    # The same planar UV projection is used for front, back and side-wall vertices.
    # This is intentionally simple: side-wall textures can stretch.
    uvs = np.vstack([uvs_front, uvs_front])

    # 2. Scale 3D coordinates to fit within the requested bounds

    scale = 0.8 / 150.0
    front_2d[:, 0] *= scale
    front_2d[:, 1] *= scale

    z_front =  0.10
    z_back  = -0.10

    # Build 3D front (0..9) and back (10..19) vertex arrays
    front_verts = np.column_stack([front_2d, np.full(10, z_front, dtype=np.float32)])
    back_verts  = np.column_stack([front_2d, np.full(10, z_back,  dtype=np.float32)])
    vertices = np.vstack([front_verts, back_verts])

    # 3. Build Triangles
    # Front fan: 9 triangles radiating from hub vertex 0 (J)
    front_tris = [
        [0, 1, 2], [0, 2, 3], [0, 3, 4], [0, 4, 5],
        [0, 5, 6], [0, 6, 7], [0, 7, 8], [0, 8, 9], [0, 9, 1]
    ]

    # Back fan: reverse winding order so normal points outward (-Z)
    back_tris = [[10 + i, 10 + k, 10 + j] for i, j, k in front_tris]

    # Side walls: quads formed along the outer loop connecting front to back
    side_tris = []
    loop = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1]
    for idx in range(len(loop) - 1):
        i = loop[idx]
        j = loop[idx + 1]
        i_back = i + 10
        j_back = j + 10
        side_tris.append([i, j, i_back])
        side_tris.append([j, j_back, i_back])

    indices = np.array(front_tris + back_tris + side_tris, dtype=np.uint32).flatten()
    return vertices, uvs, indices

# ==============================================================================
# 3. Texture Loading & GPU Allocation
# ==============================================================================

def load_texture_or_fallback(filepath):
    # TODO 2 (texture loading): inspect the Pillow image conversion, vertical flip,
    # and glTexImage2D upload below. Explain why RGB unpack alignment is 1.
    # OPTIONAL: add your own image-error handling.
    if os.path.exists(filepath):
        # Open and strictly force 3-channel RGB mode
        img = Image.open(filepath).convert("RGB")
        img = img.transpose(Image.FLIP_TOP_BOTTOM)

        # Ensure memory is contiguous C-order bytes
        img_data = np.ascontiguousarray(np.array(img, dtype=np.uint8))
        w, h = img.width, img.height
        print(f"Loaded RGB texture successfully from: {filepath} ({w}x{h})")
    else:
        print(f"Warning: '{filepath}' not found. Generating checkerboard fallback.")
        w, h = 256, 256
        grid = (np.indices((w, h)) // 32).sum(axis=0) % 2
        img_data = np.zeros((w, h, 3), dtype=np.uint8)
        img_data[grid == 0] = [220, 70, 50]
        img_data[grid == 1] = [235, 235, 240]
        img_data = np.ascontiguousarray(img_data)

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)

    # NOTE: Tell OpenGL byte rows are tightly packed with 1-byte alignment (RGB = 3 bytes/pixel)
    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)

    # Filtering & wrap modes
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    # Upload pixels as GL_RGB
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, w, h, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)
    glBindTexture(GL_TEXTURE_2D, 0)

    return tex_id

# ==============================================================================
# 4. Camera & Matrix Math
# ==============================================================================

class ArcballCamera:
    """ Mouse-driven orbit camera operating on pitch/yaw angles."""
    def __init__(self, distance=2.8):
        self.distance = distance
        self.rot_x = 20.0   # Pitch
        self.rot_y = -35.0  # Yaw
        self.last_x = 0.0
        self.last_y = 0.0
        self.is_dragging = False

    def on_press(self, x, y):
        self.is_dragging = True
        self.last_x = x
        self.last_y = y

    def on_release(self):
        self.is_dragging = False

    def on_drag(self, x, y):
        if not self.is_dragging:
            return
        dx = x - self.last_x
        dy = y - self.last_y
        self.last_x = x
        self.last_y = y

        self.rot_y += dx * 0.4
        self.rot_x += dy * 0.4
        self.rot_x = np.clip(self.rot_x, -89.0, 89.0)

    def get_view_matrix(self):
        rad_x = np.radians(self.rot_x)
        rad_y = np.radians(self.rot_y)

        Rx = np.array([
            [1.0, 0.0,            0.0,           0.0],
            [0.0, np.cos(rad_x), -np.sin(rad_x), 0.0],
            [0.0, np.sin(rad_x),  np.cos(rad_x), 0.0],
            [0.0, 0.0,            0.0,           1.0]
        ], dtype=np.float32)

        Ry = np.array([
            [ np.cos(rad_y), 0.0, np.sin(rad_y), 0.0],
            [ 0.0,           1.0, 0.0,           0.0],
            [-np.sin(rad_y), 0.0, np.cos(rad_y), 0.0],
            [ 0.0,           0.0, 0.0,           1.0]
        ], dtype=np.float32)

        T = np.identity(4, dtype=np.float32)
        T[2, 3] = -self.distance

        return T @ Rx @ Ry

def make_perspective(fov_rad, aspect, near, far):
    tan_half = np.tan(fov_rad / 2.0)
    res = np.zeros((4, 4), dtype=np.float32)
    res[0, 0] = 1.0 / (aspect * tan_half)
    res[1, 1] = 1.0 / tan_half
    res[2, 2] = -(far + near) / (far - near)
    res[2, 3] = -(2.0 * far * near) / (far - near)
    res[3, 2] = -1.0
    return res

# ==============================================================================
# 5. Shader Compilation Helper
# ==============================================================================

def compile_shader(src, stype):
    s = glCreateShader(stype)
    glShaderSource(s, src)
    glCompileShader(s)
    if not glGetShaderiv(s, GL_COMPILE_STATUS):
        raise RuntimeError(glGetShaderInfoLog(s).decode())
    return s

def build_program():
    vs = compile_shader(VERTEX_SHADER_SRC, GL_VERTEX_SHADER)
    fs = compile_shader(FRAGMENT_SHADER_SRC, GL_FRAGMENT_SHADER)
    p = glCreateProgram()
    glAttachShader(p, vs)
    glAttachShader(p, fs)
    glLinkProgram(p)
    if not glGetProgramiv(p, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(p).decode())
    glDeleteShader(vs)
    glDeleteShader(fs)
    return p

# ==============================================================================
# 6. Main Execution Loop
# ==============================================================================

def main():
    if not glfw.init():
        sys.exit("Failed to initialize GLFW.")

    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)

    window = glfw.create_window(800, 700, "HW5: Textured 3D Letter with Multiple Lights", None, None)
    if not window:
        glfw.terminate()
        sys.exit("Failed to create GLFW window.")

    glfw.make_context_current(window)
    glEnable(GL_DEPTH_TEST)

    camera = ArcballCamera(distance=2.8)

    # Mouse input handlers
    def mouse_button_cb(w, btn, act, mods):
        if btn == glfw.MOUSE_BUTTON_LEFT:
            if act == glfw.PRESS:
                x, y = glfw.get_cursor_pos(w)
                camera.on_press(x, y)
            elif act == glfw.RELEASE:
                camera.on_release()

    def cursor_pos_cb(w, x, y):
        camera.on_drag(x, y)

    glfw.set_mouse_button_callback(window, mouse_button_cb)
    glfw.set_cursor_pos_callback(window, cursor_pos_cb)

    # 1. Build and interleave geometry attributes: [x, y, z, u, v]
    verts, uvs, indices = build_textured_letter_b()
    interleaved = np.empty((len(verts), 5), dtype=np.float32)
    interleaved[:, 0:3] = verts
    interleaved[:, 3:5] = uvs

    # 2. Buffers setup (VAO, VBO, EBO)
    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, interleaved.nbytes, interleaved, GL_STATIC_DRAW)

    ebo = glGenBuffers(1)
    glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo)
    glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices.nbytes, indices, GL_STATIC_DRAW)

    stride = 5 * interleaved.itemsize
    # Location 0: aPos (vec3)
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    # Location 1: aUv (vec2)
    glVertexAttribPointer(1, 2, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(3 * 4))
    glEnableVertexAttribArray(1)

    glBindVertexArray(0)

    # 3. Load texture (look for texture.jpg or fallback)
    # TODO 3 (texture file): put a PNG/JPG next to this script and set its name.
    texture_path = os.path.join(os.getcwd(), "texture.jpg")
    tex_id = load_texture_or_fallback(texture_path)

    # 4. Build shaders and query uniform locations
    prog = build_program()
    loc_mv      = glGetUniformLocation(prog, "uModelView")
    loc_pj      = glGetUniformLocation(prog, "uPersp")
    loc_tex     = glGetUniformLocation(prog, "uTexture")
    loc_nlights = glGetUniformLocation(prog, "uNumLights")
    loc_lights  = glGetUniformLocation(prog, "uLights")

    #5.  3 static lights balanced around the model: Front-Right, Front-Left, and Back
    # TODO 4 (light placement): experiment with positions of three fixed world-space
    # lights. Keep the number of lights <= 8, as required by the shader.
    world_lights = np.array([
        [ 2.5,  2.0,  3.0],  # Light 1: Front-top-right key light
        [-2.5, -1.0,  2.0],  # Light 2: Front-bottom-left fill light
        [ 0.0,  1.5, -3.5]   # Light 3: Back light illuminating the rear face & side walls
    ], dtype=np.float32)
    num_lights = len(world_lights)

    # Render loop
    while not glfw.window_should_close(window):
        w, h = glfw.get_framebuffer_size(window)
        glViewport(0, 0, w, h)
        aspect = w / float(h if h > 0 else 1)

        glClearColor(0.12, 0.14, 0.18, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        MV = camera.get_view_matrix()
        P = make_perspective(np.radians(40.0), aspect, 0.1, 50.0)

        # Transform world-space lights into eye space using current View matrix
        # TODO 5 (coordinate transformation): transform each WORLD-space light
        # position into EYE space using MV. Use homogeneous coordinate w=1.
        # Until completed, this placeholder runs but lights will move with camera.
        eye_lights_arr = world_lights.copy().astype(np.float32)
        # Replace the line above with your loop/matrix multiplication.

        glUseProgram(prog)

        # Upload transformation matrices (transposed for GLSL column-major order)
        glUniformMatrix4fv(loc_mv, 1, GL_FALSE, MV.T.astype(np.float32))
        glUniformMatrix4fv(loc_pj, 1, GL_FALSE, P.T.astype(np.float32))

        # Upload multiple lights
        # TODO 6 (uniforms): verify how the count and array are uploaded.
        glUniform1i(loc_nlights, num_lights)
        glUniform3fv(loc_lights, num_lights, np.ascontiguousarray(eye_lights_arr))

        # Bind texture to Texture Unit 0
        glActiveTexture(GL_TEXTURE0)
        glBindTexture(GL_TEXTURE_2D, tex_id)
        glUniform1i(loc_tex, 0)

        # Render 3D textured letter
        glBindVertexArray(vao)
        glDrawElements(GL_TRIANGLES, len(indices), GL_UNSIGNED_INT, None)
        glBindVertexArray(0)

        glfw.swap_buffers(window)
        glfw.poll_events()

    # Cleanup
    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(1, [vbo])
    glDeleteBuffers(1, [ebo])
    glDeleteTextures(1, [tex_id])
    glDeleteProgram(prog)
    glfw.terminate()

if __name__ == "__main__":
    main()